In [5]:
# Import packages and GDrive Driver for Colab to access file
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report

In [7]:
# Import data

PATH = "/content/drive/MyDrive/access_model.csv"
df = pd.read_csv(PATH)
df.head()

,fips,state,county,population,pop_density,rural_urban_code,poverty_rate,median_income,unemployment_rate,uninsured_rate,...,physician_office_rate_clean,nurse_practitioner_rate_clean,physician_assistant_rate_clean,fqhc_rate_clean,rhc_rate_clean,dist_clinic_clean,dist_ed_clean,nhsc_provider_rate_clean,hpsa_score,is_hpsa
0,1001,Alabama,Autauga County,59285,99.73,2,11.7,69841.0,2.2,7.36,...,0.40,0.48,0.08,0.03,0.03,8.96,5.29,0.02,15.0,1
1,1007,Alabama,Bibb County,22152,35.59,1,19.4,51215.0,2.5,8.32,...,0.32,1.37,0.23,0.41,0.14,3.88,8.34,0.09,17.0,1
2,1009,Alabama,Blount County,59292,91.94,1,12.8,61096.0,2.1,10.19,...,0.28,0.35,0.02,0.02,0.15,4.04,9.70,0.00,11.0,1
3,1023,Alabama,Choctaw County,12525,13.71,9,24.8,44483.0,4.0,8.17,...,0.00,0.57,0.00,0.32,0.48,4.80,10.84,0.08,14.0,1
4,1027,Alabama,Clay County,14188,23.49,9,16.9,51852.0,2.4,8.42,...,0.28,0.43,0.00,0.07,0.00,5.32,6.78,0.07,11.0,1


## **Feature Engineering**

## **Modelling**

In [8]:
# baseline: predict everyone has the average shortage rate
baseline_prob = df["is_hpsa"].mean()
df["baseline_pred"] = baseline_prob

print("Baseline shortage rate:", baseline_prob)

Baseline shortage rate: 0.2167741935483871


###**Logistic Regression**

In [9]:
features = [
    "pop_density",
    "rural_urban_code",
    "poverty_rate",
    "median_income",
    "unemployment_rate",
    "uninsured_rate",
    "medicaid_rate",
    "physician_office_rate_clean",
    "nurse_practitioner_rate_clean",
    "physician_assistant_rate_clean",
    "fqhc_rate_clean",
    "rhc_rate_clean",
    "dist_clinic_clean",
    "dist_ed_clean",
    "nhsc_provider_rate_clean"
]

X = df[features]
y = df["is_hpsa"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, probs >= 0.51))

AUC: 0.7769769357495881
              precision    recall  f1-score   support

           0       0.90      0.67      0.77       607
           1       0.39      0.74      0.51       168

    accuracy                           0.69       775
   macro avg       0.65      0.71      0.64       775
weighted avg       0.79      0.69      0.72       775



###**XGBoost**

In [10]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

xgb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

probs_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGB AUC:", roc_auc_score(y_test, probs_xgb))

XGB AUC: 0.7903526319918411


## **Export Data For Tableau and GitHub**



In [12]:
df["expected_shortage_risk"] = xgb_model.predict_proba(df[features])[:, 1]

# ----------------------------
# 1. Create risk tiers
# ----------------------------
df["risk_tier"] = pd.cut(
    df["expected_shortage_risk"],
    bins=[-np.inf, 0.30, 0.50, 0.75, np.inf],
    labels=["Low", "Moderate", "Elevated", "High"]
)

# ----------------------------
# 2. Create clearer access alignment categories
# ----------------------------
def classify_access_alignment(row):
    expected = row["expected_shortage_risk"]
    official = row["is_hpsa"]

    if official == 1:
        return "Officially Designated"
    elif official == 0 and expected >= 0.50:
        return "High-Risk Not Designated"
    else:
        return "Not Designated (Lower Expected Risk)"

df["access_alignment"] = df.apply(classify_access_alignment, axis=1)

# ----------------------------
# 3. Optional: create priority flag
# ----------------------------
df["priority_review_flag"] = np.where(
    df["access_alignment"] == "High-Risk Not Designated",
    1,
    0
)

# ----------------------------
# 4. Create access gap for ranking
# ----------------------------
# Positive = officially designated more than model expected
# Negative = model expected more risk than official designation
df["access_gap"] = df["is_hpsa"] - df["expected_shortage_risk"]

df["access_gap_index"] = (
    (df["access_gap"] - df["access_gap"].min()) /
    (df["access_gap"].max() - df["access_gap"].min()) * 100
).round(1)

# ----------------------------
# 5. Optional: readable HPSA label
# ----------------------------
df["official_shortage_designation"] = np.where(
    df["is_hpsa"] == 1,
    "Yes",
    "No"
)

# ----------------------------
# 6. Export for Tableau
# ----------------------------
tableau_cols = [
    "fips",
    "state",
    "county",
    "population",
    "pop_density",
    "poverty_rate",
    "median_income",
    "unemployment_rate",
    "uninsured_rate",
    "medicaid_rate",
    "physician_office_rate_clean",
    "nurse_practitioner_rate_clean",
    "physician_assistant_rate_clean",
    "fqhc_rate_clean",
    "rhc_rate_clean",
    "dist_clinic_clean",
    "dist_ed_clean",
    "nhsc_provider_rate_clean"
    "hpsa_score",
    "is_hpsa",
    "official_shortage_designation",
    "expected_shortage_risk",
    "risk_tier",
    "access_gap",
    "access_gap_index",
    "access_alignment",
    "priority_review_flag"
]

tableau_cols = [c for c in tableau_cols if c in df.columns]

df_tableau = df[tableau_cols].copy()

OUT_DIR = "/content/drive/MyDrive"
tableau_path = f"{OUT_DIR}/healthcare_access_final.csv"
df_tableau.to_csv(tableau_path, index=False)

tableau_path

print(df_tableau["access_alignment"].value_counts(normalize=True).mul(100).round(2))
print(df_tableau["access_alignment"].value_counts())

access_alignment
Not Designated (Lower Expected Risk)    76.23
Officially Designated                   21.68
High-Risk Not Designated                 2.10
Name: proportion, dtype: float64
access_alignment
Not Designated (Lower Expected Risk)    2363
Officially Designated                    672
High-Risk Not Designated                  65
Name: count, dtype: int64
